## Acquisition des graphes de réseau OSM

Téléchargement des réseaux piéton et cyclable sur le périmètre de la
Métropole Rouen Normandie via OSMnx.

**Source** : OpenStreetMap via API Overpass  
**Périmètre** : Métropole Rouen Normandie  
**Projection** : EPSG:2154 (RGF93 / Lambert-93)  
**Sorties** :
- `data/reseau_pieton_MRN.gpkg` — arêtes du réseau piéton (vecteur)
- `data/reseau_pieton_MRN.graphml` — graphe piéton (topologie NetworkX)
- `data/reseau_velo_MRN.gpkg` — arêtes du réseau cyclable (vecteur)
- `data/reseau_velo_MRN.graphml` — graphe cyclable (topologie NetworkX)

**Date d'acquisition** : 2026-05-12

> Les fichiers `.graphml` conservent la topologie du graphe nécessaire
> au calcul des isochrones (notebook 03). Les fichiers `.gpkg` sont
> destinés à la visualisation et au contrôle qualité dans QGIS.

In [ ]:
import osmnx as ox
import os

ox.settings.timeout = 180
ox.settings.log_console = True
ox.settings.overpass_endpoint = "https://overpass-api.de/api"

place = "Métropole Rouen Normandie, France"

G_walk = ox.graph_from_place(place, network_type="walk")
G_walk_proj = ox.project_graph(G_walk, to_crs="EPSG:2154")
nodes_walk, edges_walk = ox.graph_to_gdfs(G_walk_proj)
edges_walk.to_file("../data/reseau_pieton_MRN.gpkg", driver="GPKG")
ox.save_graphml(G_walk_proj, filepath="../data/reseau_pieton_MRN.graphml")

G_bike = ox.graph_from_place(place, network_type="bike")
G_bike_proj = ox.project_graph(G_bike, to_crs="EPSG:2154")
nodes_bike, edges_bike = ox.graph_to_gdfs(G_bike_proj)
edges_bike.to_file("../data/reseau_velo_MRN.gpkg", driver="GPKG")
ox.save_graphml(G_bike_proj, filepath="../data/reseau_velo_MRN.graphml")

print("Fichiers générés avec succès.")

In [ ]:
import osmnx as ox

# Charger les graphes sauvegardés
G_walk = ox.load_graphml("../data/reseau_pieton_MRN.graphml")
G_bike = ox.load_graphml("../data/reseau_velo_MRN.graphml")

# Stats de base
for nom, G in [("Piéton", G_walk), ("Vélo", G_bike)]:
    stats = ox.basic_stats(G)
    print(f"\n── {nom} ──────────────────────")
    print(f"  Nœuds       : {G.number_of_nodes()}")
    print(f"  Arêtes      : {G.number_of_edges()}")
    print(f"  Longueur totale : {stats['edge_length_total']/1000:.1f} km")
    print(f"  Degré moyen : {stats['k_avg']:.2f}")
    print(f"  CRS         : {G.graph.get('crs', 'non défini')}")

In [ ]:
# Vérifier si le graphe est dirigé et compter les arêtes réciproques
print(f"Graphe piéton dirigé : {G_walk.is_directed()}")

# Compter les paires d'arêtes réciproques (A→B et B→A)
reciproques = sum(1 for u, v in G_walk.edges() if G_walk.has_edge(v, u))
print(f"Arêtes réciproques : {reciproques} / {G_walk.number_of_edges()}")

## Contrôle qualité des graphes

Rechargement des graphes depuis les fichiers `.graphml` et vérification
des indicateurs topologiques de base : nombre de nœuds et d'arêtes,
longueur totale du réseau, degré moyen et système de coordonnées.

| Indicateur        | Piéton     | Vélo       |
|-------------------|------------|------------|
| Nœuds             | 59 726     | 44 755     |
| Arêtes            | 164 046    | 106 189    |
| Longueur totale   | 11 967 km  | 10 063 km  |
| Degré moyen       | 5,49       | 4,75       |
| CRS               | EPSG:2154  | EPSG:2154  |

Le réseau piéton est plus dense que le cyclable, ce qui est cohérent :
OSMnx exclut du graphe vélo les voies sans accès cyclable déclaré dans OSM.

Le degré moyen élevé s'explique par la nature **dirigée** du graphe :
la totalité des arêtes (164 046/164 046) sont réciproques — chaque segment
est représenté dans les deux sens (A→B et B→A). Le degré effectif est
donc ~2,7, cohérent avec un réseau urbain mixte. Sans impact sur les
calculs d'isochrones.

La projection **EPSG:2154** est confirmée sur les deux graphes. ✓